In [ ]:
import numpy as np
from google.colab import drive
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/diss-datasets"

data = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient_v2.npy', allow_pickle=True).item()
y_tr = data['train']['y'].astype(int)
te = data['test']
y_te = te['y'].astype(int)
file = np.asarray(te['file'])
t0 = te['t0'].astype(int)

MERGE = 30
event_id = np.full(len(y_te), -1, dtype=int)
next_ev = 0
for f in np.unique(file):
    idx = np.where((file == f) & (y_te == 1))[0]
    if len(idx) == 0:
        continue
    order = idx[np.argsort(t0[idx])]
    starts = t0[order]
    ev_local = np.zeros(len(order), dtype=int)
    for i in range(1, len(order)):
        ev_local[i] = ev_local[i-1] + (1 if (starts[i] - starts[i-1]) > MERGE else 0)
    for local, gi in zip(ev_local, order):
        event_id[gi] = next_ev + local
    next_ev += ev_local[-1] + 1
n_events = next_ev

def sens_at_fpr(scores, targets=(0.05, 0.10, 0.20)):
    neg_scores = np.sort(scores[y_te == 0])[::-1]
    res = {}
    for tgt in targets:
        k = max(1, int(round(tgt * len(neg_scores))))
        thresh = neg_scores[k-1]
        caught = set(event_id[(y_te == 1) & (scores >= thresh)])
        res[tgt] = len(caught) / n_events
    return res, thresh

fs = 256
bands = {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 64)}

def band_power(X):
    T = X.shape[2]
    freqs = np.fft.rfftfreq(T, 1/fs)
    psd = np.abs(np.fft.rfft(X, axis=2))**2
    feats = [np.log(psd[:, :, (freqs >= lo) & (freqs < hi)].sum(axis=2) + 1e-8)
             for lo, hi in bands.values()]
    return np.concatenate(feats, axis=1)

def corr_feats(X):
    C = X.shape[1]
    iu = np.triu_indices(C, 1)
    out = np.empty((X.shape[0], len(iu[0])))
    for i in range(X.shape[0]):
        out[i] = np.corrcoef(X[i])[iu]
    return np.nan_to_num(out)

Xtr, Xte = data['train']['X'], data['test']['X']
Ftr = np.concatenate([band_power(Xtr), corr_feats(Xtr)], axis=1)
Fte = np.concatenate([band_power(Xte), corr_feats(Xte)], axis=1)

Ztr_c = np.load(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_train_emb.npy')
Zte_c = np.load(f'{SAVE_DIR}/Chronos2_CHBMIT_v2_test_emb.npy')
clf_c = RidgeClassifier().fit(Ztr_c, y_tr)
res, _ = sens_at_fpr(clf_c.decision_function(Zte_c))

Ztr_m = np.load(f'{SAVE_DIR}/MantisV2_CHBMIT_v2_train_emb.npy')
Zte_m = np.load(f'{SAVE_DIR}/MantisV2_CHBMIT_v2_test_emb.npy')
clf_m = RidgeClassifier().fit(Ztr_m, y_tr)
res_m, _ = sens_at_fpr(clf_m.decision_function(Zte_m))

print("Setup complete, Ftr/Fte/res/res_m ready")

In [ ]:
from sklearn.preprocessing import StandardScaler
from openpyxl import load_workbook
from openpyxl.styles import Font

scaler_f = StandardScaler().fit(Ftr)
Ftr_s = scaler_f.transform(Ftr)
Fte_s = scaler_f.transform(Fte)

scaler_zc = StandardScaler().fit(Ztr_c)
Ztr_c_s = scaler_zc.transform(Ztr_c)
Zte_c_s = scaler_zc.transform(Zte_c)

Xtr_aug_c = np.concatenate([Ztr_c_s, Ftr_s], axis=1)
Xte_aug_c = np.concatenate([Zte_c_s, Fte_s], axis=1)

clf_aug_c = RidgeClassifier().fit(Xtr_aug_c, y_tr)
acc_aug_c = accuracy_score(y_te, clf_aug_c.predict(Xte_aug_c))
res_aug_c, _ = sens_at_fpr(clf_aug_c.decision_function(Xte_aug_c))

scaler_zm = StandardScaler().fit(Ztr_m)
Ztr_m_s = scaler_zm.transform(Ztr_m)
Zte_m_s = scaler_zm.transform(Zte_m)

Xtr_aug_m = np.concatenate([Ztr_m_s, Ftr_s], axis=1)
Xte_aug_m = np.concatenate([Zte_m_s, Fte_s], axis=1)

clf_aug_m = RidgeClassifier().fit(Xtr_aug_m, y_tr)
acc_aug_m = accuracy_score(y_te, clf_aug_m.predict(Xte_aug_m))
res_aug_m, _ = sens_at_fpr(clf_aug_m.decision_function(Xte_aug_m))

print(f"RQ4 augmented (Chronos-2 + band+corr) accuracy on v2: {acc_aug_c:.4f}")
print(f"RQ4 augmented (MantisV2 + band+corr) accuracy on v2: {acc_aug_m:.4f}")

XLSX = "/content/drive/MyDrive/results.xlsx"
wb = load_workbook(XLSX)
bold = Font(bold=True)
sheet_name = "RQ4 retrofitting"
if sheet_name in wb.sheetnames:
    del wb[sheet_name]
ws = wb.create_sheet(sheet_name)

ws.cell(row=1, column=1, value="RQ4: retrofitting channel-awareness via feature concatenation, frozen encoder, no retraining").font = bold

headers = ["Method", "Channel handling", "Accuracy", "Sens @5% FPR", "Sens @10% FPR", "Sens @20% FPR"]
for c, h in enumerate(headers, 1):
    ws.cell(row=3, column=c, value=h).font = bold

rows = [
    ("Chronos-2 (baseline)", "channel-aware, no retrofit", 0.8027, res[0.05], res[0.10], res[0.20]),
    ("Chronos-2 + band+corr", "retrofitted", acc_aug_c, res_aug_c[0.05], res_aug_c[0.10], res_aug_c[0.20]),
    ("MantisV2 (baseline)", "channel-independent, no retrofit", 0.7605, res_m[0.05], res_m[0.10], res_m[0.20]),
    ("MantisV2 + band+corr", "retrofitted", acc_aug_m, res_aug_m[0.05], res_aug_m[0.10], res_aug_m[0.20]),
]
for r, row in enumerate(rows, 4):
    for c, val in enumerate(row, 1):
        ws.cell(row=r, column=c, value=val)

wb.save(XLSX)
print("saved to", XLSX)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

def box(x, y, w, h, text, subtext, color):
    b = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05,rounding_size=0.15",
                        linewidth=1, edgecolor='black', facecolor=color)
    ax.add_patch(b)
    ax.text(x + w/2, y + h*0.62, text, ha='center', va='center', fontsize=10, fontweight='bold')
    if subtext:
        ax.text(x + w/2, y + h*0.28, subtext, ha='center', va='center', fontsize=8)

def arrow(x1, y1, x2, y2):
    a = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>', mutation_scale=15,
                         linewidth=1, color='black')
    ax.add_patch(a)

box(0.5, 8, 3.5, 1.2, "Frozen encoder", "Chronos-2 or MantisV2", "#E6F1FB")
box(6, 8, 3.5, 1.2, "Explicit features", "Band power + correlation", "#FAECE7")

arrow(2.25, 8, 2.25, 6.9)
arrow(7.75, 8, 7.75, 6.9)

box(0.5, 5.7, 3.5, 1.1, "Per-window embedding", "", "#E6F1FB")
box(6, 5.7, 3.5, 1.1, "Per-window features", "", "#FAECE7")

arrow(2.25, 5.7, 4.3, 4.6)
arrow(7.75, 5.7, 5.7, 4.6)

box(3.2, 3.5, 3.6, 1.1, "Concatenate + standardise", "", "#EEEDFE")

arrow(5, 3.5, 5, 2.5)

box(2.7, 1.4, 4.6, 1.1, "Refit linear probe", "Encoder weights unchanged", "#E1F5EE")

arrow(5, 1.4, 5, 0.4)

ax.text(5, 0.2, "Event-level sensitivity at fixed FPR", ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/rq4_pipeline_diagram.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("Chronos-2 + band+corr sensitivity:", res_aug_c)
print("MantisV2 + band+corr sensitivity:", res_aug_m)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

XLSX = "/content/drive/MyDrive/results.xlsx"
df_rq3 = pd.read_excel(XLSX, sheet_name="RQ3 event sensitivity", header=2)

methods = df_rq3["Method"].tolist()
fpr_cols = ["Sens @5% FPR", "Sens @10% FPR", "Sens @20% FPR"]
x = np.arange(len(fpr_cols))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
for i, method in enumerate(methods):
    vals = df_rq3.loc[df_rq3["Method"] == method, fpr_cols].values.flatten()
    ax.bar(x + i * width, vals, width, label=method)

ax.set_xticks(x + width * (len(methods) - 1) / 2)
ax.set_xticklabels(["5% FPR", "10% FPR", "20% FPR"])
ax.set_ylabel("Event-level sensitivity")
ax.set_title("RQ3: event-level sensitivity at fixed interictal FPR")
ax.legend(loc="lower right", fontsize=8)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/rq3_sensitivity_plot.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

XLSX = "/content/drive/MyDrive/results.xlsx"
raw = pd.read_excel(XLSX, sheet_name="CHB-MIT ablations", header=None)

df_ablation = raw.iloc[3:8, [0, 1, 2]].copy()
df_ablation.columns = ["Model", "Condition", "Accuracy"]
df_ablation["Accuracy"] = df_ablation["Accuracy"].astype(float)

df_classical = raw.iloc[12:18, [0, 1]].copy()
df_classical.columns = ["Feature set", "Accuracy"]
df_classical["Accuracy"] = df_classical["Accuracy"].astype(float)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

chronos_rows = df_ablation[df_ablation["Model"] == "Chronos-2"]
axes[0].bar(chronos_rows["Condition"], chronos_rows["Accuracy"], color="#378ADD")
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel("Accuracy")
axes[0].set_title("RQ1: Chronos-2 accuracy under channel ablations")
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(chronos_rows["Accuracy"]):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha='center', fontsize=9)

colors = ["#888780"] + ["#1D9E75"] * (len(df_classical) - 1)
axes[1].barh(df_classical["Feature set"], df_classical["Accuracy"], color=colors)
axes[1].set_xlim(0, 1.0)
axes[1].set_xlabel("Accuracy")
axes[1].set_title("Classical features vs foundation model embedding")
axes[1].axvline(0.8027, color="#D85A30", linestyle="--", linewidth=1)
axes[1].text(0.8027 + 0.01, -0.6, "Chronos-2\nfull embedding", color="#D85A30", fontsize=8, va='top')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/rq1_ablation_and_classical_plot.png', dpi=300, bbox_inches='tight')
plt.show()